# 02 — Análise estatística dos sensores

Nesta etapa será investigado como os sinais variam entre motores e ao longo dos ciclos. O objetivo não será escolher um modelo, mas compreender quais informações estão presentes nos dados e quais sinais parecem redundantes.

A análise será concentrada no FD001, que possui uma única condição operacional principal. Os demais subconjuntos serão utilizados posteriormente para estudar generalização entre condições e modos de falha.

In [1]:
from pathlib import Path
import sys
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'vscode'

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import (
    correlation_matrix, load_split, operating_settings_profile,
    sensor_profile, sensor_variability_by_unit,
)
from multilang import LangMap

lang = LangMap(source='pt', target='pt')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'
fd001 = load_split(DATA_DIR, 'train', 'FD001')


In [2]:
fd001

,unit,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640


## 1. Estatística descritiva

A tabela apresenta a escala e a dispersão dos sensores. O desvio-padrão mostra quanto os valores se afastam, em média, da média do sensor. Como as unidades dos sensores são diferentes, essa tabela é descritiva; ela não deve ser interpretada como comparação direta de importância.

In [2]:
profile_fd001 = sensor_profile(fd001)
profile_fd001

,feature,missing,unique_values,minimum,maximum,mean,std
0,s1,0,1,518.6700,518.6700,518.670000,0.000000e+00
1,s10,0,1,1.3000,1.3000,1.300000,0.000000e+00
2,s16,0,1,0.0300,0.0300,0.030000,3.469531e-18
3,s18,0,1,2388.0000,2388.0000,2388.000000,0.000000e+00
4,s19,0,1,100.0000,100.0000,100.000000,0.000000e+00
5,s5,0,1,14.6200,14.6200,14.620000,5.329200e-15
6,setting3,0,1,100.0000,100.0000,100.000000,0.000000e+00
7,s6,0,2,21.6000,21.6100,21.609803,1.388985e-03
8,s17,0,13,388.0000,400.0000,393.210654,1.548763e+00
9,setting2,0,13,-0.0006,0.0006,0.000002,2.930621e-04


**Insights:** Não foram encontrados valores ausentes no FD001. A tabela também confirma que existem sete variáveis constantes e algumas com poucas categorias, como s6 e s17, enquanto outras apresentam grande variedade de valores, como s9, s14 e s21. Essa diferença descreve a forma dos dados, mas não mede sozinha a importância para a previsão de RUL: uma variável com muitos valores pode ser pouco útil, e uma variável com poucos valores pode representar uma condição operacional importante.

## 2. Variabilidade entre motores

A variabilidade dentro de cada motor é comparada com a variabilidade entre motores. Em linguagem simples, será observado se um sensor muda principalmente durante a vida de um motor ou se também apresenta diferenças relevantes entre motores diferentes. Essa distinção será importante quando a validação por trajetória for iniciada.

In [3]:
variability_fd001 = sensor_variability_by_unit(fd001)
variability_fd001

,feature,median_within_unit_std,mean_within_unit_std,between_unit_std
0,s9,7.666567,14.189053,1.272360e+01
1,s14,7.205621,11.804007,1.197666e+01
2,s4,8.060340,8.118497,3.716790e+00
3,s3,5.753704,5.788760,1.921935e+00
4,s17,1.435693,1.450997,5.123618e-01
5,s7,0.774687,0.786467,3.893508e-01
6,s12,0.638503,0.648178,3.362171e-01
7,s2,0.459652,0.462850,1.827874e-01
8,s11,0.232640,0.237445,1.170863e-01
9,s20,0.165451,0.166420,6.820239e-02


**Insights:** s9 e s14 foram os sinais com maior combinação de variação dentro dos motores e diferença entre motores. Eles podem carregar informação dinâmica e também características específicas de cada unidade, o que os torna candidatos naturais para uma investigação preditiva posterior. Essa observação não constitui evidência de causalidade nem garante desempenho no modelo. As sete variáveis constantes permaneceram sem variação dentro e entre os motores e, por isso, poderão ser excluídas do primeiro baseline.

## 3. Correlação entre sinais

A correlação de Pearson mede se dois sinais tendem a aumentar ou diminuir juntos (Pearson, 1895). Ela não demonstra que um sensor causa o comportamento do outro; apenas resume uma associação linear observada nos dados. Neste caso, a matriz será utilizada para identificar sinais potencialmente redundantes e orientar uma futura comparação de atributos.

In [4]:
corr_fd001 = correlation_matrix(fd001)
fig = px.imshow(corr_fd001, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                aspect='auto', labels={'color': 'Correlação'},
                title=lang({'title': 'Correlação entre configurações e sensores — FD001'})['title'])
fig.show()

**Insights:** A associação mais forte observada foi entre s9 e s14, com correlação de aproximadamente 0,96. Também foram observadas associações intensas entre s11 e s12, em sentido contrário, e entre outros sinais ligados ao mesmo padrão de operação. Isso sugere que alguns sensores podem estar registrando partes semelhantes de um comportamento comum, mas não permite dizer que um sensor provoca a mudança do outro. A decisão de remover ou manter sinais deverá ser tomada somente depois de uma comparação preditiva por motor, pois a redundância pode reduzir a complexidade ou, em alguns modelos, contribuir para a estabilidade.

## 4. Condições operacionais

As configurações operacionais representam o contexto em que o motor foi observado. No FD001, uma configuração é constante e as outras apresentam pouca variação. Essa característica deve ser registrada porque um modelo que funcione bem no FD001 poderá enfrentar um problema diferente quando for exposto às múltiplas condições dos conjuntos FD002 e FD004.

In [5]:
operating_settings_profile(fd001)

,feature,unique_values,minimum,maximum,mean,std
0,setting1,158,-0.0087,0.0087,-0.000009,0.002187
1,setting2,13,-0.0006,0.0006,0.000002,0.000293
2,setting3,1,100.0000,100.0000,100.000000,0.000000


**Insights:** No FD001, setting3 permaneceu fixa em 100, enquanto setting1 e setting2 apresentaram variação pequena e limitada. Assim, este conjunto representa uma condição operacional relativamente controlada. Um resultado obtido apenas no FD001 não deverá ser apresentado como evidência de generalização para ambientes variados; essa hipótese precisará ser testada nos conjuntos FD002 e FD004, que foram construídos com múltiplas condições de operação.

## Conclusões desta etapa

A estatística descritiva permite identificar sensores constantes, sensores de baixa variação e sensores que possuem escalas muito diferentes. Essas informações serão utilizadas para preparar o baseline sem transformar uma diferença de unidade de medida em uma falsa indicação de importância.

A correlação será tratada como diagnóstico de associação, e não como evidência de causalidade. A decisão sobre manter ou remover sensores será tomada posteriormente, dentro de uma validação por motor e acompanhada de um registro explícito da decisão.

## Referências

Pearson, K. (1895). Notes on regression and inheritance in the case of two parents. *Proceedings of the Royal Society of London*, 58, 240–242.